[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/ray_optics.ipynb)

# Ray optics and optical instruments full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **ray tracer**: every ray is followed surface by surface, with the law of
reflection at a mirror and Snell's law at glass or water, and nothing else. You can move all of it yourself: an
**object** in front of a mirror, the **angle** of a ray under water, a lens's two **curves** and its glass, the
**angle of incidence** on a prism, and the **reading distance** and **camera distance** of the course's opening
question.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more, **with its sign** when the question asks for v or for the
magnification. You can type the arithmetic: `1/(1/-10 - 1/-30)`, `asind(1/1.5)`, `2*tand(48.75)`,
`sind(60)/sind(40)`. The functions `sind`, `cosd`, `tand` take degrees and `asind` gives degrees; `sin`, `cos`, `tan`,
`asin` and `sqrt` work in radians, with `pi`. For several answers, separate them with commas, in the order the
question lists them (the box's label shows the order). You can also put your answer between the quotes in a cell's
last line, `practice("P1.2", "")`, and run the cell: it checks straight away.

**The sign convention** (the same everywhere in the course): light travels from left to right; distances are measured
from the mirror's pole or the lens's centre, positive along the light and negative against it; heights are positive
above the axis. So an object in front has a negative u; a concave mirror has a negative f; a convex lens a positive f.

**The cheat sheet.** Every formula and value from the course on two pages:
[ray optics cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/ray_optics_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the formulas, the ray tracer and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # charges and lines in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# rays CHALK · glass and water SKY · mirrors CHALK · dashed back-lines DIM · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="traced by the laws of reflection and refraction · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
INF = math.inf
N_AIR, N_WATER, N_GLASS, N_CROWN, N_DIAMOND = 1.0, 1.33, 1.5, 1.52, 2.42   # refractive indices
N_CORE, N_CLAD = 1.47, 1.46        # a single-mode glass fibre's core and cladding
D_NEAR = 25.0                      # cm, the near point of a normal eye, as the course takes it
C = 3.00e8                         # m/s, the speed of light

def sind(d): return math.sin(math.radians(d))
def cosd(d): return math.cos(math.radians(d))
def tand(d): return math.tan(math.radians(d))
def asind(x): return math.degrees(math.asin(x))
def _inv(k): return INF if abs(k) < 1e-15 else 1.0 / k
def close(a, b, rel=1e-4):
    """True when a traced value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---------- the course's formulas (Cartesian signs) ----------
def mirror_f(R): return R / 2                                                    # f = R/2
def mirror_v(u, f): return _inv(1 / f - 1 / u)                                   # 1/v + 1/u = 1/f
def mirror_m(u, v): return -v / u                                                # m = −v/u
def refract(theta1, n1, n2):                                                     # n₁ sin θ₁ = n₂ sin θ₂ (None: no refracted ray)
    s = n1 * sind(theta1) / n2
    return None if s > 1 else asind(s)
def critical(n1, n2=1.0): return asind(n2 / n1)                                  # sin θc = n₂/n₁ (from the slower medium n₁)
def apparent_depth(d, n): return d / n                                           # seen from air, looking straight down
def slab_shift(t, i, n):                                                         # a ray's sideways shift through a slab
    r2 = refract(i, 1.0, n); return t * sind(i - r2) / cosd(r2)
def surface_v(u, R, n1, n2): return n2 * _inv((n2 - n1) / R + n1 / u)            # n₂/v − n₁/u = (n₂ − n₁)/R
def lens_maker(n, R1, R2, n_out=1.0): return _inv((n / n_out - 1) * (1 / R1 - 1 / R2))   # 1/f = (n − 1)(1/R₁ − 1/R₂)
def lens_v(u, f): return _inv(1 / f + 1 / u)                                     # 1/v − 1/u = 1/f
def lens_m(u, v): return v / u                                                   # m = v/u
def power(f_m): return 1 / f_m                                                   # P = 1/f, f in metres: dioptres
def combined_f(f1, f2): return _inv(1 / f1 + 1 / f2)                             # lenses in contact: 1/f = 1/f₁ + 1/f₂
def prism_dev(i, A, n):                                                          # δ = i + e − A (None: no ray leaves)
    r1 = asind(sind(i) / n); r2 = A - r1
    if n * sind(r2) > 1: return None
    return i + asind(n * sind(r2)) - A
def min_dev(A, n): return 2 * asind(n * sind(A / 2)) - A                         # the least deviation
def prism_n(A, Dm): return sind((A + Dm) / 2) / sind(A / 2)                      # n = sin((A + Dm)/2) / sin(A/2)
def thin_prism(A, n): return (n - 1) * A                                         # δ = (n − 1) A
def magnifier(f, D=D_NEAR, relaxed=True): return D / f if relaxed else 1 + D / f   # a simple microscope
def microscope(fo, fe, L, D=D_NEAR): return (L / fo) * (D / fe)                  # m = (L/f_o)(D/f_e)
def telescope(fo, fe): return fo / fe                                            # m = f_o/f_e

# ---- the course's ray tracer: the law of reflection and Snell's law, surface by surface ----
# An element is a surface the ray can meet: arc(x0, R, h, n_left, n_right) is a spherical surface standing on the axis
# (vertex at x0, centre of curvature at x0 + R, used up to height h); seg(p0, p1, n_left, n_right) is a flat face;
# mirror=True makes it reflect. trace() follows one ray: the nearest surface ahead, its normal, then the law.
def seg(p0, p1, n_left=1.0, n_right=1.0, mirror=False, name=""):
    return {"kind": "seg", "p0": (float(p0[0]), float(p0[1])), "p1": (float(p1[0]), float(p1[1])), "nl": n_left,
            "nr": n_right, "mirror": mirror, "name": name}
def arc(x0, R, h, n_left=1.0, n_right=1.0, mirror=False, name="", hole=0.0):
    if R == INF: return seg((x0, -h), (x0, h), n_left, n_right, mirror, name)
    return {"kind": "arc", "x0": float(x0), "R": float(R), "h": float(h), "hole": float(hole), "nl": n_left, "nr": n_right,
            "mirror": mirror, "name": name}
def mirror(x0, R, h, name="mirror", hole=0.0): return arc(x0, R, h, mirror=True, name=name, hole=hole)
def sag(R, h): return 0.0 if R == INF else math.copysign(abs(R) - math.sqrt(R * R - h * h), R)
def lens(x0, R1, R2, n, h, n_out=1.0, edge=0.0, name="lens"):
    """A real lens centred at x0: two spherical faces (radii R1 then R2, Cartesian signs), index n, half-height h. A lens
    thicker in the middle is just thick enough to leave `edge` at its rim; one thinner in the middle is `edge` thick at
    its centre. Returns its two surfaces."""
    bulge = sag(R1, h) - sag(R2, h)
    t = bulge + edge if bulge > 0 else edge
    return [arc(x0 - t / 2, R1, h, n_out, n, name=name + ":1"), arc(x0 + t / 2, R2, h, n, n_out, name=name + ":2")]
def polygon(pts, n_in, n_out=1.0, name="glass"):
    """A closed block of glass: its corners in anticlockwise order (inside on the left of each side)."""
    return [seg(p, q, n_in, n_out, name=f"{name}:{k}") for k, (p, q) in enumerate(zip(pts, pts[1:] + pts[:1]))]

_EPS = 1e-9
def _hit(el, p, d):
    """Where the ray p + t d first meets element el: (t, point, unit normal, index on the normal's side, index on the
    other side), or None."""
    if el["kind"] == "seg":
        (x0, y0), (x1, y1) = el["p0"], el["p1"]
        ex, ey = x1 - x0, y1 - y0
        den = d[0] * ey - d[1] * ex
        if abs(den) < 1e-15: return None
        t = ((x0 - p[0]) * ey - (y0 - p[1]) * ex) / den
        s = ((x0 - p[0]) * d[1] - (y0 - p[1]) * d[0]) / den
        if t <= _EPS or s < -1e-12 or s > 1 + 1e-12: return None
        L = math.hypot(ex, ey)
        return t, (p[0] + t * d[0], p[1] + t * d[1]), (-ey / L, ex / L), el["nl"], el["nr"]
    cx, R, rho = el["x0"] + el["R"], el["R"], abs(el["R"])
    fx, fy = p[0] - cx, p[1]
    b = fx * d[0] + fy * d[1]; disc = b * b - (fx * fx + fy * fy - rho * rho)
    if disc < 0: return None
    for t in sorted((-b - math.sqrt(disc), -b + math.sqrt(disc))):
        if t <= _EPS: continue
        x, y = p[0] + t * d[0], p[1] + t * d[1]
        if el["hole"] - 1e-12 <= abs(y) <= el["h"] + 1e-12 and (x - cx) * R < 0:   # on the vertex's side of the centre, in the aperture
            N = ((x - cx) / rho, y / rho)                                 # pointing away from the centre of curvature
            return (t, (x, y), N, el["nl"], el["nr"]) if R > 0 else (t, (x, y), N, el["nr"], el["nl"])
    return None

def trace(elements, p, d, hits=40, run=0.0):
    """Follow one ray from point p in direction d through the elements. Returns {"pts": the start and every point where
    it met a surface (and, with run > 0, a last point `run` further on), "d": its final unit direction, "hits": for each
    meeting {"el", "name", "kind": "reflect" | "refract" | "tir", "i": the angle of incidence (degrees, from the normal),
    "r": the angle of reflection or refraction, "n1", "n2"}}."""
    L = math.hypot(*d); d = (d[0] / L, d[1] / L); p = (float(p[0]), float(p[1]))
    pts, out = [p], []
    for _ in range(hits):
        best = None
        for k, el in enumerate(elements):
            h = _hit(el, p, d)
            if h and (best is None or h[0] < best[1][0]): best = (k, h)
        if best is None: break
        k, (t, q, N, n_side, n_far) = best
        el = elements[k]
        if d[0] * N[0] + d[1] * N[1] > 0: N = (-N[0], -N[1]); n1, n2 = n_far, n_side      # N now faces the incoming ray
        else: n1, n2 = n_side, n_far
        ci = min(1.0, -(d[0] * N[0] + d[1] * N[1]))                        # cos of the angle of incidence
        ai = math.degrees(math.acos(ci))
        kk = -1.0 if el["mirror"] else 1 - (n1 / n2) ** 2 * (1 - ci * ci)
        if kk < 0:                                                         # the law of reflection (a mirror, or total internal reflection)
            d = (d[0] + 2 * ci * N[0], d[1] + 2 * ci * N[1])
            out.append({"el": k, "name": el["name"], "kind": "reflect" if el["mirror"] else "tir", "i": ai,
                        "r": math.degrees(math.acos(min(1.0, d[0] * N[0] + d[1] * N[1]))), "n1": n1, "n2": n1})
        else:                                                              # Snell's law, as vectors
            m = n1 / n2
            d = (m * d[0] + (m * ci - math.sqrt(kk)) * N[0], m * d[1] + (m * ci - math.sqrt(kk)) * N[1])
            out.append({"el": k, "name": el["name"], "kind": "refract", "i": ai,
                        "r": math.degrees(math.acos(min(1.0, -(d[0] * N[0] + d[1] * N[1])))), "n1": n1, "n2": n2})
        L = math.hypot(*d); d = (d[0] / L, d[1] / L)
        p = q; pts.append(q)
    if run > 0: pts.append((p[0] + run * d[0], p[1] + run * d[1]))
    return {"pts": pts, "d": d, "hits": out, "run": run}

def aim(p, q): return (q[0] - p[0], q[1] - p[1])
def last(tr):
    """The ray's final straight line: (its last meeting point, its direction)."""
    return tr["pts"][-2 if tr["run"] > 0 else -1], tr["d"]
def axis_x(tr, y=0.0):
    """Where the ray's final straight line crosses the height y (ahead of its last meeting point, or behind it: a
    virtual crossing): (x, ahead?)."""
    (px, py), (dx, dy) = last(tr)
    t = (y - py) / dy
    return px + t * dx, t > 0
def meet(tr1, tr2):
    """Where the final straight lines of two rays cross: ((x, y), real?). real: the crossing is ahead of both rays'
    last meeting points (light really passes through it); otherwise the rays only seem to come from it (virtual)."""
    (p1, d1), (p2, d2) = last(tr1), last(tr2)
    den = d1[0] * d2[1] - d1[1] * d2[0]
    t = ((p2[0] - p1[0]) * d2[1] - (p2[1] - p1[1]) * d2[0]) / den
    s = ((p2[0] - p1[0]) * d1[1] - (p2[1] - p1[1]) * d1[0]) / den
    return (p1[0] + t * d1[0], p1[1] + t * d1[1]), (t > 0 and s > 0)
def image(elements, obj, targets, run=0.0):
    """The image of the point obj: rays aimed from it at each target point are traced, and their final lines crossed in
    pairs. Returns {"pt": the mean crossing, "real": bool, "spread": the largest gap between two crossings (how far the
    rays are from meeting in one point), "rays": the traces}."""
    rays = [trace(elements, obj, aim(obj, q), run=run) for q in targets]
    xs = [meet(a, b) for i, a in enumerate(rays) for b in rays[i + 1:]]
    pt = (sum(x[0][0] for x in xs) / len(xs), sum(x[0][1] for x in xs) / len(xs))
    spread = max(math.hypot(a[0][0] - b[0][0], a[0][1] - b[0][1]) for a in xs for b in xs)
    assert len({x[1] for x in xs}) == 1, "the rays disagree on real or virtual"
    return {"pt": pt, "real": xs[0][1], "spread": spread, "rays": rays}
def laws_hold(tr, tol=1e-7):
    """Every meeting on a traced ray obeys its law: reflection (the two angles equal) or Snell's law."""
    return all((abs(h["i"] - h["r"]) < 1e-6) if h["kind"] != "refract"
               else abs(h["n1"] * sind(h["i"]) - h["n2"] * sind(h["r"])) < tol for h in tr["hits"])
def angle_of(tr): return math.degrees(math.atan2(tr["d"][1], tr["d"][0]))
def kinds(tr): return [h["kind"] for h in tr["hits"]]

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "cos": math.cos, "sin": math.sin, "tan": math.tan, "asin": math.asin, "acos": math.acos,
         "atan": math.atan, "sind": sind, "cosd": cosd, "tand": tand, "asind": asind}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, pi, sqrt, sin, cos, tan, asin (radians) and sind,
    cosd, tand, asind (degrees)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = r"\s*(m/s|cm|mm|m|D|dioptres?|°|degrees?|times|×)\s*$"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Mirrors',
     "q": 'A concave mirror has a radius of curvature of 36 cm. How far from the mirror is its focus, in cm? (Give the distance, without a sign.)',
     "parts": ['focal length (cm)'],
     "answer": [18.0], "tol": [0.005], "function": '—',
     "hint": 'the focus is halfway between the mirror and its centre of curvature',
     "working": 'f = R/2 = 36 / 2 = 18 cm'},
    {"id": 'P1.2', "tier": 1, "group": 'Mirrors',
     "q": 'An object stands 30 cm in front of a concave mirror of focal length 10 cm. With the sign convention (u = −30 cm, f = −10 cm), what is v, in cm?',
     "parts": ['v (cm)'],
     "answer": [-14.999999999999996], "tol": [0.005], "function": '—',
     "hint": '1/v + 1/u = 1/f, with u and f negative',
     "working": '1/v = 1/f − 1/u = −1/10 + 1/30 = −2/30, so v = −15 cm: 15 cm in front of the mirror, a real image'},
    {"id": 'P1.3', "tier": 1, "group": 'Mirrors',
     "q": 'For that mirror (u = −30 cm, v = −15 cm), what is the magnification, with its sign?',
     "parts": ['magnification'],
     "answer": [-0.5], "tol": [0.005], "function": '—',
     "hint": 'for a mirror, m = −v/u',
     "working": 'm = −v/u = −(−15)/(−30) = −0.5: half the size, upside down'},
    {"id": 'P1.4', "tier": 1, "group": 'Refraction',
     "q": 'Light travels at 3.00 × 10⁸ m/s in air. How fast does it travel in glass of refractive index 1.5, in units of 10⁸ m/s?',
     "parts": ['speed (× 10⁸ m/s)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'the refractive index says how many times slower: v = c ÷ n',
     "working": 'v = c/n = 3.00 × 10⁸ / 1.5 = 2.00 × 10⁸ m/s'},
    {"id": 'P1.5', "tier": 1, "group": 'Refraction',
     "q": 'A ray in air meets a water surface (n = 1.33) at 30° to the normal. What is the angle of refraction, in degrees?',
     "parts": ['angle of refraction (°)'],
     "answer": [22.082413194472252], "tol": [0.005], "function": 'sin, asin',
     "hint": 'n₁ sin θ₁ = n₂ sin θ₂',
     "working": 'sin θ₂ = sin 30° / 1.33 = 0.3759, so θ₂ = 22.08°'},
    {"id": 'P1.6', "tier": 1, "group": 'Refraction',
     "q": 'What is the critical angle for glass of refractive index 1.5, in air, in degrees?',
     "parts": ['critical angle (°)'],
     "answer": [41.810314895778596], "tol": [0.005], "function": 'asin',
     "hint": 'sin θc = n₂/n₁, here 1 ÷ 1.5',
     "working": 'sin θc = 1/1.5 = 0.6667, so θc = 41.81°'},
    {"id": 'P1.7', "tier": 1, "group": 'Lenses',
     "q": 'An object stands 60 cm in front of a convex lens of focal length 20 cm (u = −60 cm, f = +20 cm). What is v, in cm?',
     "parts": ['v (cm)'],
     "answer": [29.999999999999993], "tol": [0.005], "function": '—',
     "hint": '1/v − 1/u = 1/f',
     "working": '1/v = 1/f + 1/u = 1/20 − 1/60 = 2/60, so v = +30 cm: a real image 30 cm behind the lens'},
    {"id": 'P1.8', "tier": 1, "group": 'Lenses',
     "q": 'A convex lens has a focal length of 40 cm. What is its power, in dioptres?',
     "parts": ['power (D)'],
     "answer": [2.5], "tol": [0.005], "function": '—',
     "hint": 'power = 1 ÷ the focal length in metres',
     "working": 'P = 1/f = 1 / 0.40 m = 2.5 D'},
    {"id": 'P1.9', "tier": 1, "group": 'Prism and instruments',
     "q": 'A thin glass prism (n = 1.5) has an angle of 4°. By how many degrees does it turn a ray?',
     "parts": ['deviation (°)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'for a thin prism, δ = (n − 1) A',
     "working": 'δ = (n − 1)A = 0.5 × 4° = 2°'},
    {"id": 'P1.10', "tier": 1, "group": 'Prism and instruments',
     "q": 'A telescope has an objective of focal length 80 cm and an eyepiece of focal length 4 cm. What is its magnification?',
     "parts": ['magnification'],
     "answer": [20.0], "tol": [0.005], "function": '—',
     "hint": "m = the objective's focal length ÷ the eyepiece's",
     "working": 'm = f_o/f_e = 80 / 4 = 20'},
    {"id": 'P2.1', "tier": 2, "group": 'Mirrors',
     "q": 'An object stands 5 cm in front of a concave mirror of focal length 15 cm. What is v (cm, with its sign), and what is the magnification?',
     "parts": ['v (cm)', 'magnification'],
     "answer": [7.499999999999998, 1.4999999999999996], "tol": [0.005, 0.005], "function": '—',
     "hint": 'inside the focus, v comes out positive: behind the mirror',
     "working": '1/v = −1/15 + 1/5 = 2/15, so v = +7.5 cm: behind the mirror, a virtual image; m = −v/u = −7.5/(−5) = +1.5: upright, one and a half times the size'},
    {"id": 'P2.2', "tier": 2, "group": 'Mirrors',
     "q": "A car's convex side mirror has a focal length of 25 cm. A cyclist is 100 cm from it. What is v (cm), and what is the magnification?",
     "parts": ['v (cm)', 'magnification'],
     "answer": [20.0, 0.2], "tol": [0.005, 0.005], "function": '—',
     "hint": 'a convex mirror has a positive f; u is still negative',
     "working": '1/v = 1/25 + 1/100 = 5/100, so v = +20 cm (behind the mirror); m = −20/(−100) = +0.2: upright, a fifth of the size'},
    {"id": 'P2.3', "tier": 2, "group": 'Refraction',
     "q": 'A tank holds water (n = 1.33) to a depth of 80 cm. Looking straight down, how deep does it seem, in cm?',
     "parts": ['apparent depth (cm)'],
     "answer": [60.150375939849624], "tol": [0.005], "function": '—',
     "hint": 'seen from air, straight down: real depth ÷ n',
     "working": 'apparent depth = real depth / n = 80 / 1.33 = 60.15 cm'},
    {"id": 'P2.4', "tier": 2, "group": 'Refraction',
     "q": 'A ray meets a glass slab (n = 1.5) 4 cm thick at 60° to the normal. By how much is it shifted sideways when it leaves, in cm?',
     "parts": ['shift (cm)'],
     "answer": [2.0498880527646595], "tol": [0.005], "function": 'sin, cos, asin',
     "hint": 'find the angle inside first; the path inside is t / cos r long',
     "working": 'inside, sin r = sin 60°/1.5, r = 35.26°; shift = t sin(i − r)/cos r = 4 × sin 24.74° / cos 35.26° = 2.05 cm'},
    {"id": 'P2.5', "tier": 2, "group": 'Refraction',
     "q": "A small lamp sits 1.5 m below the surface of a pool (n = 1.33). Its light leaves the water only through a circle on the surface. What is the circle's radius, in metres?",
     "parts": ['radius (m)'],
     "answer": [1.7106309628511807], "tol": [0.005], "function": 'asin, tan',
     "hint": 'beyond the critical angle the light is all reflected back down',
     "working": "at the circle's edge the rays meet the surface at the critical angle, 48.75°; radius = 1.5 × tan 48.75° = 1.71 m"},
    {"id": 'P2.6', "tier": 2, "group": 'Lenses',
     "q": 'The end of a glass rod (n = 1.5) is rounded, bulging towards the air with a radius of 10 cm. A small lamp sits on the axis, 40 cm in front of it (u = −40 cm, R = +10 cm). Where is the image, v, in cm?',
     "parts": ['v (cm)'],
     "answer": [60.0], "tol": [0.005], "function": '—',
     "hint": 'one curved surface: n₂/v − n₁/u = (n₂ − n₁)/R',
     "working": 'n₂/v − n₁/u = (n₂ − n₁)/R: 1.5/v = 0.5/10 − 1/40 = 0.025, so v = +60 cm, inside the glass'},
    {"id": 'P2.7', "tier": 2, "group": 'Lenses',
     "q": 'A glass lens (n = 1.5) bulges on both sides, with radii of 15 cm and 30 cm (R₁ = +15 cm, R₂ = −30 cm). What is its focal length, in cm?',
     "parts": ['focal length (cm)'],
     "answer": [20.0], "tol": [0.005], "function": '—',
     "hint": "the lens-maker's formula; the second face's R is negative",
     "working": '1/f = (n − 1)(1/R₁ − 1/R₂) = 0.5 × (1/15 + 1/30) = 0.05, so f = 20 cm'},
    {"id": 'P2.8', "tier": 2, "group": 'Lenses',
     "q": 'An object stands 30 cm in front of a concave lens of focal length 15 cm (f = −15 cm). What is v (cm), and what is the magnification?',
     "parts": ['v (cm)', 'magnification'],
     "answer": [-10.0, 0.3333333333333333], "tol": [0.005, 0.005], "function": '—',
     "hint": 'a concave lens has a negative f; for a lens m = v/u',
     "working": '1/v = 1/f + 1/u = −1/15 − 1/30 = −3/30, so v = −10 cm (the same side as the object: virtual); m = v/u = −10/(−30) = +0.33: upright, a third of the size'},
    {"id": 'P2.9', "tier": 2, "group": 'Prism and instruments',
     "q": 'A prism of angle 60° is made of glass of refractive index 1.6. What is its angle of minimum deviation, in degrees?',
     "parts": ['minimum deviation (°)'],
     "answer": [46.260204708311946], "tol": [0.005], "function": 'sin, asin',
     "hint": 'at the minimum the ray passes symmetrically: r = A/2',
     "working": 'at minimum deviation r = A/2 = 30°, so sin i = 1.6 × sin 30° = 0.8, i = 53.13°; D = 2i − A = 46.26°'},
    {"id": 'P2.10', "tier": 2, "group": 'Glasses and camera',
     "q": "A reader's near point has moved out to 75 cm. What power of reading glasses lets them read a page held at 25 cm, in dioptres?",
     "parts": ['power (D)'],
     "answer": [2.666666666666667], "tol": [0.005], "function": '—',
     "hint": 'work in metres, and 1/f is the power: 1/v − 1/u',
     "working": 'the lens must make, of the page at u = −25 cm, a virtual image at v = −75 cm: 1/f = 1/v − 1/u = −1/0.75 + 1/0.25 = 2.67 per metre, so P = +2.67 D'},
    {"id": 'P3.1', "tier": 3, "group": 'Mirrors',
     "q": 'A concave mirror of focal length 10 cm is to throw a real image four times the size of the object. How far in front of the mirror must the object stand, in cm? (Give the distance, without a sign.)',
     "parts": ['object distance (cm)'],
     "answer": [12.5], "tol": [0.005], "function": '—',
     "hint": 'real images are upside down, so m = −4; then v = 4u in the mirror formula',
     "working": 'a real image is upside down: m = −4 = −v/u, so v = 4u; 1/(4u) + 1/u = 1/f gives 5/(4u) = −1/10, u = −12.5 cm (and v = −50 cm)'},
    {"id": 'P3.2', "tier": 3, "group": 'Mirrors',
     "q": "In a convex side mirror, a car 5 m away gives an image 1 m behind the mirror. What is the mirror's focal length (m), and its radius of curvature (m)?",
     "parts": ['focal length (m)', 'radius of curvature (m)'],
     "answer": [1.25, 2.5], "tol": [0.005, 0.005], "function": '—',
     "hint": 'run the mirror formula backwards; R = 2f',
     "working": 'u = −5 m, v = +1 m: 1/f = 1/v + 1/u = 1 − 0.2 = 0.8, so f = +1.25 m and R = 2f = 2.5 m'},
    {"id": 'P3.3', "tier": 3, "group": 'Refraction',
     "q": 'Light travels inside glass (n = 1.5) towards a surface with water (n = 1.33) on the other side. Beyond what angle of incidence is it totally reflected, in degrees?',
     "parts": ['critical angle (°)'],
     "answer": [62.4573248455412], "tol": [0.005], "function": 'asin',
     "hint": 'the second medium is water, not air: n₂/n₁',
     "working": "sin θc = n₂/n₁ = 1.33/1.5 = 0.8867, so θc = 62.46° (much more than glass to air's 41.8°)"},
    {"id": 'P3.4', "tier": 3, "group": 'Refraction',
     "q": "A ray in air meets a liquid at 60° to the normal and refracts at 40°. What is the liquid's refractive index?",
     "parts": ['refractive index'],
     "answer": [1.3472963553338608], "tol": [0.005], "function": 'sin',
     "hint": "run Snell's law backwards: n = sin θ₁ ÷ sin θ₂",
     "working": 'n = sin 60° / sin 40° = 0.8660 / 0.6428 = 1.35'},
    {"id": 'P3.5', "tier": 3, "group": 'Lenses',
     "q": 'A glass lens (n = 1.5, both faces of radius 20 cm) has a focal length of 20 cm in air. What is its focal length under water (n = 1.33), in cm?',
     "parts": ['focal length (cm)'],
     "answer": [78.23529411764703], "tol": [0.005], "function": '—',
     "hint": 'only the ratio of the two indices matters: replace n by 1.5/1.33',
     "working": "in water the lens-maker's formula uses n_glass/n_water − 1 = 1.5/1.33 − 1 = 0.1278: 1/f = 0.1278 × (1/20 + 1/20), so f = 78.24 cm: almost four times as long"},
    {"id": 'P3.6', "tier": 3, "group": 'Lenses',
     "q": 'A convex lens of focal length 10 cm and a concave lens of focal length 25 cm are placed in contact. What is the power of the pair (D), and its focal length (cm)?',
     "parts": ['power (D)', 'focal length (cm)'],
     "answer": [6.0, 16.666666666666668], "tol": [0.005, 0.005], "function": '—',
     "hint": "powers in contact simply add; the concave lens's power is negative",
     "working": 'P = P₁ + P₂ = 1/0.10 − 1/0.25 = 10 − 4 = +6 D; f = 1/6 m = 16.67 cm'},
    {"id": 'P3.7', "tier": 3, "group": 'Prism and instruments',
     "q": 'A compound microscope has an objective of focal length 4 mm and a tube length of 160 mm. What eyepiece focal length gives a magnification of 400, with the near point at 25 cm? (In cm.)',
     "parts": ['eyepiece focal length (cm)'],
     "answer": [2.5], "tol": [0.005], "function": '—',
     "hint": "m = (L/f_o) × (D/f_e); find the objective's share first",
     "working": 'the objective magnifies L/f_o = 160/4 = 40 times, so the eyepiece must give 400/40 = 10 = D/f_e: f_e = 25/10 = 2.5 cm'},
    {"id": 'P3.8', "tier": 3, "group": 'Glasses and camera',
     "q": "A phone camera's lens has a focal length of 7.05 mm. Set for a far scene, it sits 7.05 mm from the sensor. How far must it move away from the sensor to focus on a flower 20 cm in front of it (mm), and how tall is the image of the 5 cm flower on the sensor (mm, as a size without a sign)?",
     "parts": ['lens movement (mm)', 'image height (mm)'],
     "answer": [0.2575926405804614, 1.8268981601451153], "tol": [0.005, 0.005], "function": '—',
     "hint": 'work in millimetres; for a far scene v = f',
     "working": '1/v = 1/7.05 − 1/200, so v = 7.3076 mm: the lens moves 0.26 mm; m = v/u = 7.308/(−200) = -0.03654, so the image is 1.83 mm tall, upside down'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert round(mirror_v(-60.0, -20.0), 9) == -30.0 and round(lens_v(-30.0, 10.0), 9) == 15.0
print("Ready: n(water) =", N_WATER, "· n(glass) =", N_GLASS, "· near point D =", D_NEAR, "cm")

## Tier 1 · Warm-up

One idea, one step: a mirror's focal length, the mirror formula and its magnification, the speed of light in glass,
Snell's law, the critical angle, the thin lens formula, the power of a lens, a thin prism and a telescope.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `mirror_v(u, f)`, `mirror_m(u, v)`,
`refract(theta1, n1, n2)`, `critical(n1, n2)`, `lens_v(u, f)`, `lens_maker(n, R1, R2)`, `power(f_in_metres)`,
`thin_prism(A, n)`, `telescope(fo, fe)` and the rest. Keep the signs of the convention (an object in front: negative
u). Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.2  v = {mirror_v(-30, -10):.2f} cm;  P1.5  angle = {refract(30, 1, N_WATER):.2f}°;  P1.6  critical = {critical(N_GLASS):.2f}°")
assert agrees("P1.1", [abs(mirror_f(-36))]) and agrees("P1.2", [mirror_v(-30, -10)]) and agrees("P1.3", [mirror_m(-30, -15)])
assert agrees("P1.4", [C / N_GLASS / 1e8]) and agrees("P1.5", [refract(30, 1, N_WATER)]) and agrees("P1.6", [critical(N_GLASS)])
assert agrees("P1.7", [lens_v(-60, 20)]) and agrees("P1.8", [power(0.40)]) and agrees("P1.9", [thin_prism(4, 1.5)])
assert agrees("P1.10", [telescope(80, 4)])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## A mirror's image you can move

An arrow stands in front of a spherical mirror. Three rays leave its tip: one parallel to the axis, one along the line
through the centre of curvature, one aimed at the pole. Each is traced to the mirror and reflected there about the
normal, which is the line to the centre of curvature. Where the reflected rays meet (or, traced back, seem to come
from) is the image. The title compares that traced point with the mirror formula. Heights are stretched in the
picture, because the formula is for rays close to the axis. Slide the object inside the focus and watch the image jump
behind the mirror; make R positive for a convex mirror.

In [ ]:
def mirror_picture(u_cm=-60.0, R_cm=-40.0, h=1.2, half=3.5):
    f = mirror_f(R_cm); v = mirror_v(u_cm, f); m = mirror_m(u_cm, v)
    els = [mirror(0.0, R_cm, half)]
    y_c = h + (0.0 - u_cm) * (0.0 - h) / (R_cm - u_cm) if abs(R_cm - u_cm) > 1e-9 else 0.0
    y_c = max(-half * 0.98, min(half * 0.98, y_c))
    img = image(els, (u_cm, h), [(sag(R_cm, h), h), (sag(R_cm, y_c), y_c), (0.0, 0.0)])
    fig, ax = plt.subplots(figsize=(10, 4.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.14)
    ys = np.linspace(-half, half, 60)
    ax.plot([sag(R_cm, abs(y)) for y in ys], ys, color=CHALK, lw=3)                    # the mirror
    ax.axhline(0, color=GHOST, lw=1)
    for tr in img["rays"]:
        (x0, y0), (x1, y1) = tr["pts"][0], tr["pts"][1]
        ax.plot([x0, x1], [y0, y1], color=CHALK, lw=1.6)
        if img["real"]: ax.plot([x1, img["pt"][0]], [y1, img["pt"][1]], color=CHALK, lw=1.6)
        else:
            ax.plot([x1, x1 + 30 * tr["d"][0]], [y1, y1 + 30 * tr["d"][1]], color=CHALK, lw=1.6)
            ax.plot([x1, img["pt"][0]], [y1, img["pt"][1]], color=DIM, lw=1.2, ls="--")
    ax.annotate("", (u_cm, h), (u_cm, 0), arrowprops=dict(arrowstyle="->", color=SKY, lw=2.5))
    ax.annotate("", img["pt"], (img["pt"][0], 0), arrowprops=dict(arrowstyle="->", color=GOLD, lw=2.5, ls="-" if img["real"] else "--"))
    for name, x in (("P", 0.0), ("F", f), ("C", R_cm)): ax.plot([x], [0], "o", color=DIM, ms=4); ax.text(x, -0.35, name, color=DIM, ha="center", va="top")
    lim = max(abs(u_cm), abs(R_cm), abs(img["pt"][0])) * 1.1
    ax.set_xlim(-lim, lim * (0.45 if img["real"] else 1.0)); ax.set_ylim(-max(4.0, abs(img["pt"][1]) * 1.2), max(4.0, abs(img["pt"][1]) * 1.2))
    ax.set_xlabel("cm along the axis (heights stretched)"); ax.grid(False)
    kind = "real, in front" if img["real"] else "virtual, behind"
    fig.suptitle(f"traced: v = {img['pt'][0]:.2f} cm, m = {img['pt'][1] / h:.3f} ({kind}) · formula: v = {v:.2f} cm, m = {m:.3f}",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return img["pt"][0], v, img["pt"][1] / h, m, img["real"]

v_t, v_f, m_t, m_f, real_ = mirror_picture(-60.0, -40.0)
assert close(v_t, v_f, 5e-3) and close(m_t, m_f, 5e-3) and real_ and round(v_f, 9) == -30      # within half a percent of the mirror formula
v_t, v_f, m_t, m_f, real_ = mirror_picture(-10.0, -40.0, h=1.0)
assert close(v_t, v_f, 5e-3) and not real_ and round(v_f, 9) == 20 and round(m_f, 9) == 2      # inside the focus: virtual, upright, twice the size
v_t, v_f, m_t, m_f, real_ = mirror_picture(-30.0, 40.0, h=1.5)
assert close(v_t, v_f, 5e-3) and not real_ and round(v_f, 9) == 12                              # a convex mirror: always virtual

In [ ]:
interact(mirror_picture, u_cm=FloatSlider(value=-60.0, min=-100.0, max=-5.0, step=5.0, description="u (cm)"),
         R_cm=FloatSlider(value=-40.0, min=-60.0, max=60.0, step=20.0, description="R (cm)"),
         h=FloatSlider(value=1.2, min=0.5, max=1.5, step=0.1, description="height (cm)"),
         half=FloatSlider(value=3.5, min=3.5, max=3.5, step=0.1, description="mirror (cm)"));

## Tier 2 · Core

The level of the course's worked problems: a virtual image in a concave mirror, a convex side mirror, apparent depth,
a slab's sideways shift, the circle of light above an underwater lamp, one curved surface, the lens-maker's formula,
a concave lens, a prism's minimum deviation, and a pair of reading glasses.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

Two-step problems are two lines. For the slab, find the angle inside the glass first; for the lamp, the critical angle
first.

In [ ]:
v = mirror_v(-5, -15); assert agrees("P2.1", [v, mirror_m(-5, v)])
v = mirror_v(-100, 25); assert agrees("P2.2", [v, mirror_m(-100, v)])
assert agrees("P2.3", [apparent_depth(80, N_WATER)]) and agrees("P2.4", [slab_shift(4, 60, 1.5)])
print(f"P2.5  radius = {1.5 * tand(critical(N_WATER)):.2f} m;  P2.9  least deviation = {min_dev(60, 1.6):.2f}°")
assert agrees("P2.5", [1.5 * tand(critical(N_WATER))]) and agrees("P2.6", [surface_v(-40, 10, 1.0, 1.5)])
assert agrees("P2.7", [lens_maker(1.5, 15, -30)])
v = lens_v(-30, -15); assert agrees("P2.8", [v, lens_m(-30, v)])
assert agrees("P2.9", [min_dev(60, 1.6)]) and agrees("P2.10", [1 / 0.25 - 1 / 0.75])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## Snell's law and the critical angle

A ray comes up from under the surface of a slower material (water, glass, diamond: set its refractive index) and
meets the surface at the angle you choose. It is traced with Snell's law. Below the critical angle it leaves, bent
away from the normal; a step past it, Snell's law has no answer and the whole ray reflects: total internal reflection.
The title shows n₁ sin θ₁ and n₂ sin θ₂, which must be equal whenever a ray gets out.

In [ ]:
def snell_picture(angle=40.0, n_below=1.33, n_above=1.0):
    surf = [seg((-12.0, 0.0), (12.0, 0.0), n_above, n_below)]
    tr = trace(surf, (-6 * tand(angle), -6.0), (sind(angle), cosd(angle)), run=7.0)
    hit = tr["hits"][0]
    crit = critical(n_below, n_above) if n_below > n_above else None
    fig, ax = plt.subplots(figsize=(7.6, 5.2))
    fig.subplots_adjust(left=0.06, right=0.98, top=0.84, bottom=0.06)
    ax.fill_between([-12, 12], -7, 0, color=SKY, alpha=0.18); ax.axhline(0, color=CHALK, lw=1.5)
    ax.plot([0, 0], [-7, 7], color=DIM, lw=1, ls="--")                                 # the normal
    xs, ys = zip(*tr["pts"]); ax.plot(xs, ys, color=GOLD if hit["kind"] == "tir" else CHALK, lw=2.4)
    ax.set_xlim(-12, 12); ax.set_ylim(-7, 7); ax.set_aspect("equal"); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
    if hit["kind"] == "refract":
        t = f"{angle:g}° in, {hit['r']:.1f}° out · n₁ sin θ₁ = {hit['n1'] * sind(hit['i']):.4f} = n₂ sin θ₂ = {hit['n2'] * sind(hit['r']):.4f}"
    else:
        t = f"{angle:g}° is past the critical angle ({crit:.1f}°): total internal reflection"
    fig.suptitle(t, x=0.03, ha="left", color=GOLD)
    footnote(fig, "traced with Snell's law"); plt.show()
    return hit["kind"], hit["r"], crit

kind_, out_, crit_ = snell_picture(40.0)
assert kind_ == "refract" and close(N_WATER * sind(40.0), sind(out_), 1e-9) and round(crit_, 1) == 48.8
kind_, out_, crit_ = snell_picture(50.0)
assert kind_ == "tir" and close(out_, 50.0, 1e-9)                       # past 48.8°: it reflects, at an equal angle
assert snell_picture(24.0, n_below=2.42)[0] == "refract" and snell_picture(25.0, n_below=2.42)[0] == "tir"   # diamond: 24.4°
assert agrees("P1.6", [snell_picture(30.0, n_below=1.5)[2]]) and agrees("P3.3", [critical(N_GLASS, N_WATER)])

In [ ]:
interact(snell_picture, angle=FloatSlider(value=40.0, min=0.0, max=85.0, step=1.0, description="angle (°)"),
         n_below=FloatSlider(value=1.33, min=1.1, max=2.5, step=0.01, description="n below"),
         n_above=FloatSlider(value=1.0, min=1.0, max=1.4, step=0.01, description="n above"));

## A lens built from two curved surfaces

A real lens: two spherical faces of glass, nothing "thin" about the tracing. Parallel rays close to the axis are bent
by Snell's law at the first face, cross the glass, and are bent again at the second. Where they cross the axis is the
focus. The title compares it with the lens-maker's formula, 1/f = (n − 1)(1/R₁ − 1/R₂). Change the curves, the glass,
and the liquid round the lens (water: 1.33) and watch the focus move.

In [ ]:
def lens_picture(R1_cm=10.0, R2_cm=-10.0, n=1.5, n_around=1.0, half=0.6):
    els = lens(0.0, R1_cm, R2_cm, n, half, n_out=n_around)
    f = lens_maker(n, R1_cm, R2_cm, n_around)
    rays = [trace(els, (-0.6 * abs(f) - 2, y), (1.0, 0.0), run=1.7 * abs(f)) for y in (0.5, 0.3, 0.15, -0.15, -0.3, -0.5)]
    cross = [axis_x(tr)[0] for tr in rays]
    fig, ax = plt.subplots(figsize=(10, 3.8))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.16)
    ys = np.linspace(-half, half, 40)
    for el in els: ax.plot([el["x0"] + sag(el["R"], abs(y)) for y in ys], ys, color=SKY, lw=2)
    ax.axhline(0, color=GHOST, lw=1)
    for tr in rays:
        xs, yy = zip(*tr["pts"]); ax.plot(xs, yy, color=CHALK, lw=1.4)
    ax.plot([sum(cross) / len(cross)], [0], "o", color=GOLD, ms=9)
    ax.set_xlabel("cm along the axis (heights stretched)"); ax.grid(False)
    fig.suptitle(f"traced focus: {sum(cross) / len(cross):.3f} cm · lens-maker's formula: f = {f:.3f} cm · power {100 / f:+.2f} D",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return cross, f

cross_, f_ = lens_picture(10.0, -10.0, 1.5)
assert round(f_, 9) == 10 and all(close(x, f_, 5e-3) for x in cross_)            # every ray within half a percent of 10 cm
cross_, f_ = lens_picture(10.0, -10.0, 1.5, n_around=1.33)
assert round(f_, 1) == 39.1 and all(close(x, f_, 5e-3) for x in cross_)          # the same lens under water: almost four times as long
cross_, f_ = lens_picture(15.0, -30.0, 1.5)
assert agrees("P2.7", [f_]) and all(close(x, f_, 5e-3) for x in cross_)
assert agrees("P3.5", [lens_maker(1.5, 20, -20, 1.33)])

In [ ]:
interact(lens_picture, R1_cm=FloatSlider(value=10.0, min=5.0, max=40.0, step=1.0, description="R₁ (cm)"),
         R2_cm=FloatSlider(value=-10.0, min=-40.0, max=-5.0, step=1.0, description="R₂ (cm)"),
         n=FloatSlider(value=1.5, min=1.4, max=1.9, step=0.01, description="n (glass)"),
         n_around=FloatSlider(value=1.0, min=1.0, max=1.35, step=0.01, description="n (around)"),
         half=FloatSlider(value=0.6, min=0.6, max=0.6, step=0.1, description="half-height"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: a mirror that must enlarge four times, a side mirror's
radius, glass against water, a liquid's refractive index from two angles, a lens under water, two lenses in contact,
a microscope's eyepiece, and the phone camera.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards is one line of algebra, then one line of Python. For the camera, work in millimetres: set
for a far scene the lens sits one focal length from the sensor.

In [ ]:
assert agrees("P3.1", [abs(-10 * (-4 - 1) / -4)]) and agrees("P3.2", [1 / (1 / 1 + 1 / -5), 2 / (1 / 1 + 1 / -5)])
assert agrees("P3.3", [critical(N_GLASS, N_WATER)]) and agrees("P3.4", [sind(60) / sind(40)])
assert agrees("P3.5", [lens_maker(1.5, 20, -20, 1.33)]) and agrees("P3.6", [power(0.10) + power(-0.25), 100 / (power(0.10) + power(-0.25))])
assert agrees("P3.7", [D_NEAR / (400 / (16 / 0.4))])
v_mm = lens_v(-200, 7.05)
print(f"P3.8  the lens moves {v_mm - 7.05:.2f} mm; the flower's image is {abs(lens_m(-200, v_mm)) * 50:.2f} mm tall")
assert agrees("P3.8", [v_mm - 7.05, abs(lens_m(-200, v_mm)) * 50])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## A prism's deviation curve

A ray meets one face of a glass prism at the angle you choose, is traced through both faces with Snell's law, and
leaves turned towards the base. The right-hand graph is the angle it was turned through, for every angle of incidence:
it falls to a lowest value and rises again. At the lowest point the ray crosses the prism level with its base, and
n = sin((A + D)/2) / sin(A/2).

In [ ]:
def prism_ray(i, A=60.0, n=1.5, side=6.0, run=5.0):
    half, hgt = side * sind(A / 2), side * cosd(A / 2)
    els = polygon([(-half, 0.0), (half, 0.0), (0.0, hgt)], n)
    a = -A / 2 + i; d = (cosd(a), sind(a)); mid = (-half / 2, hgt / 2)
    return els, trace(els, (mid[0] - 5 * d[0], mid[1] - 5 * d[1]), d, run=run)

def traced_dev(i, A=60.0, n=1.5):
    _, tr = prism_ray(i, A, n)
    if [h["kind"] for h in tr["hits"]] != ["refract", "refract"]: return None
    (x0, y0), (x1, y1) = tr["pts"][0], tr["pts"][1]
    return math.degrees(math.atan2(y1 - y0, x1 - x0)) - math.degrees(math.atan2(tr["d"][1], tr["d"][0]))

def prism_picture(i=48.6, A=60.0, n=1.5):
    els, tr = prism_ray(i, A, n)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    fig.subplots_adjust(left=0.04, right=0.98, top=0.82, bottom=0.14, wspace=0.18)
    ax = axes[0]
    px = [e["p0"][0] for e in els] + [els[0]["p0"][0]]; py = [e["p0"][1] for e in els] + [els[0]["p0"][1]]
    ax.fill(px, py, color=SKY, alpha=0.18); ax.plot(px, py, color=SKY, lw=2)
    xs, ys = zip(*tr["pts"]); ax.plot(xs, ys, color=GOLD, lw=2.4)
    ax.set_aspect("equal"); ax.set_xlim(-8, 8); ax.set_ylim(-2, 7); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
    angles = np.arange(20.0, 88.0, 0.5); devs = [traced_dev(x, A, n) for x in angles]
    axes[1].plot([x for x, d in zip(angles, devs) if d is not None], [d for d in devs if d is not None], color=CHALK)
    dv = traced_dev(i, A, n)
    if dv is not None: axes[1].plot([i], [dv], "o", color=GOLD, ms=9)
    axes[1].set_xlabel("angle of incidence (°)"); axes[1].set_ylabel("deviation (°)")
    t = (f"deviation {dv:.2f}° (formula i + e − A: {prism_dev(i, A, n):.2f}°) · least possible: {min_dev(A, n):.2f}°" if dv is not None
         else "no ray leaves the second face: total internal reflection")
    fig.suptitle(t, x=0.03, ha="left", color=GOLD)
    footnote(fig, "traced with Snell's law at both faces"); plt.show()
    return dv

d_min = prism_picture((60 + min_dev(60, 1.5)) / 2)
assert close(d_min, min_dev(60, 1.5), 1e-9) and round(d_min, 1) == 37.2 and close(prism_n(60, d_min), 1.5, 1e-9)
assert all(traced_dev(x) >= d_min - 1e-9 for x in range(30, 86, 5)) and close(traced_dev(40.0), prism_dev(40.0, 60.0, 1.5), 1e-9)
assert prism_picture(25.0) is None and agrees("P2.9", [min_dev(60, 1.6)]) and agrees("P1.9", [thin_prism(4, 1.5)])

In [ ]:
interact(prism_picture, i=FloatSlider(value=48.6, min=20.0, max=85.0, step=0.2, description="incidence (°)"),
         A=FloatSlider(value=60.0, min=30.0, max=70.0, step=5.0, description="prism angle (°)"),
         n=FloatSlider(value=1.5, min=1.3, max=1.8, step=0.01, description="n"));

## The reading glasses and the phone camera

The course's opening question, to play with. **Glasses:** a reader's own near point has moved out; the lens must make
a page held closer seem to sit at that near point: 1/f = 1/v − 1/u with both distances negative, and the power is 1/f
in metres. **Camera:** a lens of focal length 7.05 mm; for a far scene it sits 7.05 mm from the sensor, and for
anything nearer it must move out. Slide the distances and read the answers off the titles.

In [ ]:
def glasses_picture(near_point_cm=50.0, read_at_cm=25.0):
    f = _inv(1 / -near_point_cm - 1 / -read_at_cm); Pw = power(f / 100)
    els = lens(0.0, f, -f, N_GLASS, 1.5)                                   # a real lens of that focal length
    img = image(els, (-read_at_cm, 0.5), [(0.0, 1.0), (0.0, 0.0), (0.0, -0.6)])
    fig, ax = plt.subplots(figsize=(10, 3.6))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.8, bottom=0.17)
    ax.axhline(0, color=GHOST, lw=1); ax.plot([0, 0], [-1.5, 1.5], color=SKY, lw=3)
    for tr in img["rays"]:
        xs, ys = zip(*tr["pts"]); ax.plot(xs, ys, color=CHALK, lw=1.5)
        (x1, y1) = tr["pts"][-1]; ax.plot([x1, x1 + 25 * tr["d"][0]], [y1, y1 + 25 * tr["d"][1]], color=CHALK, lw=1.5)
        ax.plot([x1, img["pt"][0]], [y1, img["pt"][1]], color=DIM, lw=1.2, ls="--")
    ax.plot([-read_at_cm], [0.5], "o", color=SKY, ms=8); ax.plot([img["pt"][0]], [img["pt"][1]], "o", color=GOLD, ms=9)
    ax.set_xlabel("cm from the lens (heights stretched)"); ax.grid(False)
    fig.suptitle(f"page at {read_at_cm:g} cm, near point {near_point_cm:g} cm: f = {f:.1f} cm, power {Pw:+.2f} D · traced: the page "
                 f"seems {-img['pt'][0]:.1f} cm away", x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return Pw, -img["pt"][0]

Pw_, seen_ = glasses_picture(50.0, 25.0)
assert round(Pw_, 9) == 2 and close(seen_, 50.0, 5e-3)                     # plus two dioptres; the page seems 50 cm away
assert agrees("P2.10", [glasses_picture(75.0, 25.0)[0]])

In [ ]:
interact(glasses_picture, near_point_cm=FloatSlider(value=50.0, min=30.0, max=200.0, step=5.0, description="near point (cm)"),
         read_at_cm=FloatSlider(value=25.0, min=20.0, max=40.0, step=1.0, description="read at (cm)"));

In [ ]:
def camera_picture(distance_cm=50.0, f_mm=7.05, height_cm=20.0):
    u = -distance_cm * 10; v = lens_v(u, f_mm); m = lens_m(u, v)
    ds = np.geomspace(8.0, 1000.0, 120)
    fig, ax = plt.subplots(figsize=(10, 3.8))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.8, bottom=0.2)
    ax.semilogx(ds, [lens_v(-d * 10, f_mm) - f_mm for d in ds], color=CHALK)
    ax.plot([distance_cm], [v - f_mm], "o", color=GOLD, ms=9)
    ax.set_xlabel("distance of the subject (cm)"); ax.set_ylabel("the lens moves out by (mm)")
    fig.suptitle(f"subject at {distance_cm:g} cm: v = {v:.3f} mm, the lens moves {v - f_mm:.3f} mm · m = {m:.5f} · a {height_cm:g} cm "
                 f"subject is {abs(m) * height_cm * 10:.2f} mm tall on the sensor", x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own example · computed"); plt.show()
    return v - f_mm, abs(m) * height_cm * 10

move_, tall_ = camera_picture(50.0)
assert round(move_, 2) == 0.10 and round(tall_, 1) == 2.9                 # a tenth of a millimetre; a 20 cm face 2.9 mm tall
move_, tall_ = camera_picture(20.0, height_cm=5.0)
assert agrees("P3.8", [move_, tall_])

In [ ]:
interact(camera_picture, distance_cm=FloatSlider(value=50.0, min=8.0, max=1000.0, step=1.0, description="distance (cm)"),
         f_mm=FloatSlider(value=7.05, min=3.0, max=12.0, step=0.05, description="f (mm)"),
         height_cm=FloatSlider(value=20.0, min=1.0, max=200.0, step=1.0, description="height (cm)"));

## Sources
Every exercise and set-up here is our own. The refractive indices are the course's rounded values (water 1.33, glass
1.5, crown glass 1.52, diamond 2.42: OpenStax University Physics volume 3, table 1.1; a single-mode glass fibre's core
1.47 and cladding 1.46: Newport, "Fiber Basics"). The near point is taken as 25 cm, as textbooks do. The phone camera
is a 7.05 mm lens, worked out in the course from one phone maker's published field of view (82°) and sensor size (a
1/1.31-inch type sensor, 12.26 mm corner to corner). The ray tracer is the course's: the law of reflection and
Snell's law, on the true spherical surfaces.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph19_ray_optics/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph19_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (N_WATER, N_GLASS, N_DIAMOND, D_NEAR, C) == (_P.N_WATER, _P.N_GLASS, _P.N_DIAMOND, _P.D_NEAR, _P.C)
    _t = trace(_P.CONCAVE, (-60.0, 3.0), (1.0, 0.0), run=45.0)                 # the notebook's tracer is the course's
    assert _t["pts"] == _P.RAYS["sm_watch"]["rays"][0]["pts"]
    assert close(_P.SPECS["P"], Pw_, 1e-12) and close(_P.CAM["travel"], camera_picture(50.0, _P.CAM["f"])[0], 1e-9)
    print("matches the course's practice set")